# US Asset Selection Algorithm

This notebook implements an algorithm to select potential US assets to invest in from the entire US asset universe (NASDAQ, NYSE, AMEX).

## Strategy
The algorithm screens assets based on the following criteria:
1. **Trend**: The asset price is above its 200-day Simple Moving Average (SMA), indicating a long-term uptrend.
2. **Momentum**: A "Golden Cross" condition where the 50-day SMA is above the 200-day SMA.
3. **Valuation/Overbought**: The Relative Strength Index (RSI) is below 70, suggesting the asset is not currently overbought.


In [ ]:
# Install necessary libraries
!pip install yfinance pandas matplotlib requests beautifulsoup4 lxml textblob

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import requests
import io
import time
from textblob import TextBlob

## 1. Get List of All US Tickers
We fetch the ticker list from the NASDAQ Trader website, which includes NASDAQ, NYSE, and AMEX listings.

In [ ]:
def get_all_tickers():
    """Fetches a comprehensive list of US tickers from NASDAQ Trader."""
    url = "https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqtraded.txt"
    try:
        df = pd.read_csv(url, sep='|')
        # Filter out test issues and the file creation time footer
        df = df[df['Test Issue'] == 'N']
        if 'Symbol' in df.columns:
            tickers = df['Symbol'].tolist()
            # Clean tickers
            tickers = [str(t) for t in tickers if isinstance(t, str)]
            return tickers
        else:
            print("Could not find 'Symbol' column in the retrieved data.")
            return []
    except Exception as e:
        print(f"Error fetching all tickers: {e}")
        return []

all_tickers = get_all_tickers()
print(f"Found {len(all_tickers)} tickers.")
print(f"Sample: {all_tickers[:10]}")

## 2. Define Analysis Functions
We define functions to calculate RSI and process assets in batches using `yfinance` bulk download.

In [ ]:
def get_news_sentiment(ticker):
    """Fetches news and calculates average sentiment polarity."""
    try:
        asset = yf.Ticker(ticker)
        news = asset.news
        if not news:
            return 0, 0
        
        sentiments = []
        for item in news:
            title = item.get('title')
            if not title:
                content = item.get('content', {})
                title = content.get('title')
            
            if title:
                blob = TextBlob(title)
                sentiments.append(blob.sentiment.polarity)
        
        if not sentiments:
            return 0, 0
            
        avg_sentiment = sum(sentiments) / len(sentiments)
        
        return avg_sentiment, len(sentiments)
        
    except Exception as e:
        print(f"Error fetching news for {ticker}: {e}")
        return 0, 0

def calculate_rsi(data, window=14):
    """Calculates the Relative Strength Index (RSI)."""
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def process_single_df(df, ticker):
    """Helper to process a single asset dataframe."""
    if df.empty:
        return None
        
    # Check for 'Close' column
    if 'Close' not in df.columns:
        return None
        
    close = df['Close']
    
    # Ensure enough data points
    if len(close) < 200:
        return None
        
    # Calculate indicators
    sma_50 = close.rolling(window=50).mean()
    sma_200 = close.rolling(window=200).mean()
    rsi = calculate_rsi(close)
    
    last_close = close.iloc[-1]
    last_sma_50 = sma_50.iloc[-1]
    last_sma_200 = sma_200.iloc[-1]
    last_rsi = rsi.iloc[-1]
    
    if pd.isna(last_sma_200) or pd.isna(last_rsi):
        return None

    # Strategy Logic
    if (last_close > last_sma_200 and 
        last_rsi < 70 and 
        last_sma_50 > last_sma_200):
        
        return {
            'Ticker': ticker,
            'Close': last_close,
            'SMA_50': last_sma_50,
            'SMA_200': last_sma_200,
            'RSI': last_rsi
        }
    return None

def analyze_batch(tickers):
    """Fetches data and calculates indicators for a batch of assets."""
    if not tickers:
        return []
        
    try:
        # Download data for the last 2 years in bulk
        data = yf.download(tickers, period="2y", group_by='ticker', threads=True, progress=False)
        
        if data.empty:
            return []

        results = []
        
        if isinstance(data.columns, pd.MultiIndex):
            downloaded_tickers = data.columns.levels[0].unique()
            for ticker in downloaded_tickers:
                try:
                    df = data[ticker]
                    res = process_single_df(df, ticker)
                    if res:
                        results.append(res)
                except Exception:
                    continue
        else:
            # Single ticker
            ticker = tickers[0]
            res = process_single_df(data, ticker)
            if res:
                results.append(res)
                
        return results
        
    except Exception as e:
        print(f"Error analyzing batch: {e}")
        return []

## 3. Run Analysis
We iterate through the tickers in batches. This is more efficient and reliable than processing one by one.

In [ ]:
selected_assets = []

BATCH_SIZE = 100
# LIMITING TO FIRST 500 FOR DEMO SPEED in Colab
# Uncomment the next line to scan ALL tickers (will take time)
# tickers_to_process = all_tickers
tickers_to_process = all_tickers[:500] 

print(f"Scanning {len(tickers_to_process)} tickers...")

for i in range(0, len(tickers_to_process), BATCH_SIZE):
    batch = tickers_to_process[i:i + BATCH_SIZE]
    print(f"Processing batch {i // BATCH_SIZE + 1} ({len(batch)} tickers)...", end="\r")
    results = analyze_batch(batch)
    selected_assets.extend(results)

print(f"\nSelected {len(selected_assets)} assets.")

if selected_assets:
    print("Analyzing news sentiment for candidates...")
    for asset in selected_assets:
        ticker = asset['Ticker']
        sentiment, count = get_news_sentiment(ticker)
        asset['News Sentiment'] = sentiment
        asset['News Count'] = count

## 4. Display Results
Show the selected assets sorted by RSI.

In [ ]:
results_df = pd.DataFrame(selected_assets)

if not results_df.empty:
    columns_to_show = ['Ticker', 'Close', 'SMA_50', 'SMA_200', 'RSI', 'News Sentiment', 'News Count']
    cols = [c for c in columns_to_show if c in results_df.columns]
    display_df = results_df[cols].sort_values(by='RSI').round(2)
    print(display_df.to_string(index=False))
else:
    print("No assets matched the criteria.")

## 5. Visual Inspection
Plot the chart for the top candidate.

In [ ]:
if not results_df.empty:
    top_pick = results_df.sort_values(by='RSI').iloc[0]['Ticker']
    print(f"\nGenerating chart for top pick: {top_pick}")
    
    df = yf.download(top_pick, period="2y", progress=False)
    
    # Handle MultiIndex if present
    if isinstance(df.columns, pd.MultiIndex):
         try:
             close = df['Close'][top_pick]
         except KeyError:
             close = df['Close']
    else:
        close = df['Close']
    
    sma_50 = close.rolling(window=50).mean()
    sma_200 = close.rolling(window=200).mean()
    
    plt.figure(figsize=(12, 8))
    plt.plot(df.index, close, label='Close Price', alpha=0.5)
    plt.plot(df.index, sma_50, label='SMA 50', color='orange')
    plt.plot(df.index, sma_200, label='SMA 200', color='red')
    plt.title(f'{top_pick} Price Analysis')
    plt.xlabel('Date')
    plt.ylabel('Price')
    plt.legend()
    plt.grid(True)
    plt.show()